In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Tip: For large simulations, switch to a GPU runtime first:
#   Runtime → Change runtime type → T4 GPU (free)
# Then re-run this cell to install the CUDA-enabled JAX.
try:
    import google.colab  # noqa: F401
    import shutil

    if shutil.which("nvidia-smi"):
        %pip install -q -U softmobility "jax[cuda12]"
    else:
        %pip install -q softmobility
except ImportError:
    pass

# Example 02. Settling flexible fiber

A planar flexible fiber settles under transverse gravity.
After a transient, it converges toward a horseshoe-like steady configuration whose curvature is set by the **elasto-gravitational number**

$$\hat B = B / F_\perp L^2 ,$$
with $B$ the bending rigidity, which is $(\pi/4)E a^4$ for a beam of radius $a$ and Young's modulus $E$.

We produce two figures:

1. **Settling transient** at fixed $\hat B = 0.02$ — five snapshots of an initially straight fiber, stacked vertically as time advances.
2. **Final shape vs rigidity** for long-time-asympototic shape for four bending rigidities, $\hat B \in \{0.02, 0.01, 0.005, 0.0025\}$, also stacked vertically (stiff → floppy).

The motion is planar so we use `FlexibleFiber(planar=True)`; the fiber bends in the xz-plane and gravity acts along $-\boldsymbol{e}_3$.


In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import softmobility as sm
from softmobility.classes import figstyle

# Apply the paper-figure matplotlib aesthetic (white background, black
# box, Helvetica labels at 11 pt, fixed colour palette). Every figure
# inherits it.
figstyle.apply()
FIGDIR = "figures"

## Helper function: lab-frame bead positions

`FlowBodyRollout.rollout` returns the **body-reference** position and orientation plus the per-bead DOFs.
To plot the fiber shape we convert each bead's body-frame position into the lab frame using the body Rodrigues vector.


In [ ]:
def lab_bead_positions(fiber, positions, orientations, dofs, design=None):
    """Return an array of shape (n_steps, n_beads, 3) with lab-frame bead positions."""
    if design is None:
        design = np.asarray(fiber.design_defaults)
    n_steps = int(positions.shape[0])
    n_beads = fiber.Nspheres
    t_dummy = np.array([0.0])
    out = np.zeros((n_steps, n_beads, 3))
    body_frame_positions = np.zeros((n_beads, 3))
    for step in range(n_steps):
        dof_step = np.asarray(dofs[step])
        for i in range(n_beads):
            body_frame_positions[i] = np.asarray(fiber.spheres[i].position(dof_step, design, t_dummy))
        R = np.asarray(sm.rotation_matrix(orientations[step]))
        out[step] = np.asarray(positions[step]) + body_frame_positions @ R.T
    return out

## Note on the bending model

`FlexibleFiber` implements the Gears Model of Delmotte et al. 2015 (§2.3) with a **linearized** version of the paper's bending torque (eq. 32 + 34 expanded to first order in joint angles, §2.4).
Bond length is exactly ``2a`` (sphere surfaces touching).
The bending torque on bead $i$ is

$$ \boldsymbol\gamma^b_i \;=\; \frac{B}{2 a}\,\bigl(\mathrm{rod}_{i-1} - 2\,\mathrm{rod}_i + \mathrm{rod}_{i+1}\bigr) $$

(adjacent-bead Laplacian on the orientation DOFs; first-difference at the chain ends).
This linearization keeps the bending operator linear in the DOFs, which preserves the framework's exact `M_K @ dofs` interface and its JIT-friendly tensor extraction.

**Approximation regime and known limitation.** This linearized model is quantitatively accurate for joint angles ≲ π/4.
It does *not* fully capture very flexible fibers ($\hat B \lesssim 0.001$).

## Settling fiber

### Elasto-gravitational number

$$\hat B = B / F_\perp L^2 , \qquad F_\perp = N\,m\,g.$$

We set $m = 1$, $g = 1$, $N = 10$, and solve for $B$ given a target $\hat B$.


In [ ]:
N_set = 10
a = 1.0
m = 1.0
g = 1.0

L_set = (N_set - 1) * 2 * a  # bond length is exactly 2a (Gears Model)
F_perp = N_set * m * g


def bending_rigidity_for_B_hat(B_hat: float) -> float:
    """Bending modulus that yields elasto-gravitational number ``B_hat``."""
    return F_perp * L_set**2 * B_hat


# Build the fiber once at the B_hat = 0.02 for reference. The bending modulus
# is a design parameter, so we can override it per-rollout via ``design=``
# to sweep B without rebuilding (and re-JIT-compiling) the rollout.
B_hat_ref = 0.02
fiber_set = sm.FlexibleFiber(
    n_beads=N_set,
    radius=a,
    bending_rigidity=bending_rigidity_for_B_hat(B_hat_ref),
    mass=m,
    planar=True,
)
rollout_set = sm.FlowBodyRollout(
    soft_body=fiber_set,
    flow=sm.no_flow(),
    input_map={"gravity": sm.gravity_field(g=g)},
)
print(
    f"L = {L_set:.1f},  F_perp = {F_perp:.1f},  "
    f"B(B_hat={B_hat_ref:g}) = {bending_rigidity_for_B_hat(B_hat_ref):.4f}"
)

### Run the rollout

The fiber is initially nearly vertical (with an angle $\theta_0=31\pi/64$) and gravity pulls it in $-\boldsymbol{e}_3$.
The terminal settling speed is of the order of $V_s\!\sim\!0.1$ in our units, so a typical time is $\tau = L/V_s \sim 100$.
We integrate over a few $\tau$ the settling dynamics and the U-curl progress.

The first call triggers JAX/XLA compilation (~10–15 s); subsequent calls (used in the rigidity sweep below) are sub-second because they reuse the same JIT-compiled function and only swap the design vector.

In [ ]:
dt_set = 0.5
n_steps_set = 20000  # final time = 10000 ≈ 100 τ


@jax.jit
def run_settling(design):
    """JIT-compiled rollout. Compiled once; reused for every B in the sweep."""
    return rollout_set.rollout(
        dt=dt_set, n_steps=n_steps_set, design=design, init_orientation=[0, 31 * np.pi / 64, 0]
    )


design_ref = jnp.array([a, bending_rigidity_for_B_hat(B_hat_ref), m])
positions_g, orientations_g, dofs_g = run_settling(design_ref)
lab_pos_g = lab_bead_positions(fiber_set, positions_g, orientations_g, dofs_g, design=np.asarray(design_ref))
print("max |dof| over trajectory:", float(np.max(np.abs(dofs_g))))

### Figures for a transient settling

Five snapshots of the $\hat B = 0.02$ fiber, COM-centred and stacked vertically (top = early, bottom = late).
Beads are drawn as filled circles at the true bead radius $a$ — the visible bead size relative to the fiber length is $a / L = 1/18$.


In [ ]:
step_indices = np.array([0, 5000, 8000, 19999])
times = step_indices * dt_set
CoM = lab_pos_g.mean(axis=1) / L_set  # normalise by fiber length for better axis limits

x = CoM[:, 0]
y = CoM[:, 2]

fig, ax = figstyle.figure(size="third", aspect=1 / 2)
ax.plot(x, y, color=figstyle.COLORS["blue"], linewidth=1.5)
ax.plot(
    x[step_indices],
    y[step_indices],
    marker="o",
    linestyle="",
    markersize=6,
    markeredgecolor=figstyle.COLORS["blue"],
    markerfacecolor="white",
    markeredgewidth=1.5,
)

# color=figstyle.COLORS["blue"], marker="o", markersize=4, linestyle="none")

ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-90, 5)
ax.set_xlabel("x / L")
ax.set_ylabel("z / L")

figstyle.save(fig, "fig_settling_transient1", figdir=FIGDIR)

In [ ]:
def draw_shape(ax, beads_xz, *, color, fill, line_width=1.5, bead_radius=a, opacity=1.0):
    """Draw a fiber as a connecting line plus one filled circle per bead."""
    xs, zs = beads_xz[:, 0], beads_xz[:, 1]
    ax.plot(xs, zs, color=color, linewidth=line_width, alpha=opacity)
    for cx, cz in zip(xs, zs, strict=True):
        ax.add_patch(
            plt.Circle(
                (cx, cz),
                bead_radius,
                edgecolor=color,
                facecolor=fill,
                linewidth=1.0,
                alpha=opacity,
            )
        )


def stack_shape(snapshot_xz, row_index, z_step):
    """Centre an (n_beads, 2) snapshot at the origin, then shift down by row_index*z_step."""
    centred = snapshot_xz - snapshot_xz.mean(axis=0)
    centred[:, 1] -= row_index * z_step
    return centred


z_step = 0.75 * L_set

# Layout. Bead column extends to roughly ±0.5 L; reserve the leftmost
# 0.6 L of the canvas for labels.
x_left, x_right = -0.65 * L_set, 0.65 * L_set

fig1, ax1 = figstyle.figure(size="half", aspect=0.6)

for i, (step, _t) in enumerate(zip(step_indices, times, strict=True)):
    snap_xz = lab_pos_g[step][:, [0, 2]]  # take (x, z)
    placed = stack_shape(snap_xz, i, z_step)
    draw_shape(
        ax1,
        placed,
        color=figstyle.COLORS["red"],
        fill=figstyle.COLORS["red_25"],
        opacity=0.6,
    )

y_top = 0.8 * z_step
y_bot = -(len(step_indices) - 1) * z_step - 0.25 * z_step
ax1.set_xlim(x_left, x_right)
ax1.set_ylim(y_bot, y_top)
ax1.set_aspect("equal")
ax1.set_axis_off()

figstyle.save(fig1, "fig_settling_transient2", figdir=FIGDIR)

### Figure of final shape vs bending rigidity

Late-time configurations for four values of $\hat B$, stacked vertically (top = stiff, bottom = floppy).
At $\hat B = 0.02$ the fiber barely deforms; by $\hat B = 0.0025$ it has folded into a tight U-curl.
The most flexible cases relax slowly from a straight fiber, so we **warm-start** each rollout from the converged shape of the previous (less-flexible) $\hat B$ — this gives a converged equilibrium for every $\hat B$ at a fraction of the cost of running each one cold from the straight chain.


In [ ]:
# Warm-start chain: run B_hat values in order of increasing flexibility,
# passing each run's final state as the IC for the next. The most
# flexible cases (large B_hat) take the longest to relax from a straight
# fiber; warm-starting them from the previous (less-flexible) shape
# cuts the transient. n_steps_chain is set so each leg converges to
# Δ ~ 1e-3 a (see drafts/test_rigidity_sweep_convergence.py).
n_steps_chain = 3000


@jax.jit
def run_settling_chain(design, init_position, init_orientation, init_dofs):
    """JIT rollout that accepts a starting state for warm-start chaining."""
    return rollout_set.rollout(
        dt=dt_set,
        n_steps=n_steps_chain,
        design=design,
        init_position=init_position,
        init_orientation=init_orientation,
        init_dofs=init_dofs,
    )


B_hat_values = [0.02, 0.01, 0.005, 0.0025]
final_shapes = []
max_dof_log = []

# Cold start IC: straight fiber at origin (the framework defaults).
init_position = jnp.zeros(3)
init_orientation = jnp.zeros(3)
init_dofs = jnp.zeros(fiber_set.Ndof)

for B_hat in B_hat_values:
    design_B = jnp.array([a, bending_rigidity_for_B_hat(B_hat), m])
    pos, ori, dof = run_settling_chain(
        design_B,
        init_position,
        init_orientation,
        init_dofs,
    )
    final_shapes.append(lab_bead_positions(fiber_set, pos, ori, dof, design=np.asarray(design_B))[-1])
    max_dof_log.append((B_hat, float(np.max(np.abs(dof)))))
    # Carry the converged state forward as the IC for the next (more flexible) B_hat.
    init_position = pos[-1]
    init_orientation = ori[-1]
    init_dofs = dof[-1]

print("max |dof| per B_hat:")
for B_hat, m_dof in max_dof_log:
    print(f"  B_hat = {B_hat:>6g}    max|dof| = {m_dof:.3f}")

fig2, ax2 = figstyle.figure(size="half", aspect=0.6)

for i, (_B_hat, snap) in enumerate(zip(B_hat_values, final_shapes, strict=True)):
    snap_xz = snap[:, [0, 2]]
    placed = stack_shape(snap_xz, i, z_step)
    draw_shape(
        ax2,
        placed,
        color=figstyle.COLORS["red"],
        fill=figstyle.COLORS["red_25"],
        opacity=0.6,
    )

y_bot2 = -(len(B_hat_values) - 1) * z_step - 0.5 * z_step
ax2.set_xlim(x_left, x_right)
ax2.set_ylim(y_bot2, y_top)
ax2.set_aspect("equal")
ax2.set_axis_off()

figstyle.save(fig2, "fig_settling_rigidity_sweep", figdir=FIGDIR)

## Notes

* `FlexibleFiber` uses **rigid bonds**  plus a **linear discrete biharmonic** bending torque.
  This captures the qualitative shapes of flexible fibers but would require large $N$ to .
* The rigidity sweep uses a **warm-start chain**: each rollout starts from the converged final state of the previous $\hat B$ via the ``init_position`` / ``init_orientation`` / ``init_dofs`` arguments of ``rollout``.
  The chain JIT is compiled once and reused for every $\hat B$ in the sweep — only the design vector and the IC change between rollouts.


## References

B. Delmotte, E. E. Keaveny, F. Plouraboué, and E. Climent, Large-scale simulation of steady and time-dependent active suspensions with the force-coupling method, *J. Comp. Phys.* **302**, 524 (2015).

L. Li, H. Manikantan, D. Saintillan, and S. E. Spagnolie, The sedimentation of flexible filaments, *J. Fluid Mech.* **735**, 705 (2013).
